# Model Evaluation: ResNet50 Classifier (15 Epochs)

Testing the trained classifier on the test dataset.

In [1]:
import tensorflow as tf
from tensorflow import keras
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

# Configuration
IMAGE_SIZE = 224
BATCH_SIZE = 8
BASE_PATH = Path.cwd() / 'split_dataset'

print(f"TensorFlow version: {tf.__version__}")
print(f"Dataset path: {BASE_PATH}")
print(f"Path exists: {BASE_PATH.exists()}")

TensorFlow version: 2.16.1
Dataset path: /Users/abhishekpanthee/Documents/hackathon/ml-app/split_dataset
Path exists: True


In [ ]:
# Load the trained model
# The model was saved with Keras 2.x, but we're using Keras 3.x
# So we need to rebuild the architecture and load weights

MODEL_PATH = Path.cwd() / 'model-training' / 'resnet50_classifier_15epochs.keras'
print(f"Loading model from: {MODEL_PATH}")
print(f"Model exists: {MODEL_PATH.exists()}")

def build_classifier_model(image_size=224):
    """Rebuild the ResNet50 classifier architecture matching the original training."""
    from tensorflow.keras.applications import ResNet50
    from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
    from tensorflow.keras.models import Model
    
    # Load ResNet-50 pretrained base
    base_model = ResNet50(
        weights='imagenet',
        include_top=False,
        input_shape=(image_size, image_size, 3)
    )
    
    # Freeze all layers first, then unfreeze last 30 (matching original training)
    for layer in base_model.layers:
        layer.trainable = False
    for layer in base_model.layers[-30:]:
        layer.trainable = True
    
    # Add classification head (matching original architecture)
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(512, activation='relu')(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    x = Dense(1, activation='sigmoid')(x)
    
    model = Model(inputs=base_model.input, outputs=x, name='ResNet50_Classifier')
    return model

# Try multiple loading strategies
try:
    # Strategy 1: Direct load with safe_mode=False (may work for some Keras 3 versions)
    classifier_model = keras.models.load_model(MODEL_PATH, safe_mode=False)
    print(f"\n✅ Model loaded with safe_mode=False!")
except Exception as e1:
    print(f"⚠️ safe_mode=False failed: {str(e1)[:100]}...")
    try:
        # Strategy 2: Load with compile=False
        classifier_model = keras.models.load_model(MODEL_PATH, compile=False)
        classifier_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        print(f"\n✅ Model loaded with compile=False!")
    except Exception as e2:
        print(f"⚠️ compile=False failed: {str(e2)[:100]}...")
        try:
            # Strategy 3: Rebuild architecture and load weights from .keras file
            import zipfile
            import tempfile
            import os
            
            print("🔄 Rebuilding model architecture and loading weights...")
            classifier_model = build_classifier_model(IMAGE_SIZE)
            
            # Extract weights from .keras file (it's a zip archive)
            with tempfile.TemporaryDirectory() as tmpdir:
                with zipfile.ZipFile(MODEL_PATH, 'r') as zip_ref:
                    zip_ref.extractall(tmpdir)
                weights_path = os.path.join(tmpdir, 'model.weights.h5')
                if os.path.exists(weights_path):
                    classifier_model.load_weights(weights_path)
                    print(f"\n✅ Model rebuilt and weights loaded from .keras archive!")
                else:
                    raise FileNotFoundError("model.weights.h5 not found in .keras archive")
            
            classifier_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
        except Exception as e3:
            print(f"❌ All loading strategies failed: {e3}")
            raise

print(f"   Input shape: {classifier_model.input_shape}")
print(f"   Output shape: {classifier_model.output_shape}")
print(f"   Total params: {classifier_model.count_params():,}")

Loading model from: /Users/abhishekpanthee/Documents/hackathon/ml-app/model-training/resnet50_classifier_15epochs.keras
Model exists: True
⚠️ Standard load failed: Could not deserialize class 'Functional' because its parent module keras.src.engine.functional cannot be imported. Full object config: {'module': 'keras.src.engine.functional', 'class_name': 'Functional', 'config': {'name': 'ResNet50_Classifier', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_input_shape': [None, 224, 224, 3], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_1'}, 'registered_name': None, 'name': 'input_1', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'ZeroPadding2D', 'config': {'name': 'conv1_pad', 'trainable': False, 'dtype': 'float32', 'padding': [[3, 3], [3, 3]], 'data_format': 'channels_last'}, 'registered_name': None, 'build_config': {'input_shape': [None, 224, 224, 3]}, 'name': 'conv1_pad', 'inbound_nodes': [[

TypeError: Could not deserialize class 'Functional' because its parent module keras.src.engine.functional cannot be imported. Full object config: {'module': 'keras.src.engine.functional', 'class_name': 'Functional', 'config': {'name': 'ResNet50_Classifier', 'trainable': True, 'layers': [{'module': 'keras.layers', 'class_name': 'InputLayer', 'config': {'batch_input_shape': [None, 224, 224, 3], 'dtype': 'float32', 'sparse': False, 'ragged': False, 'name': 'input_1'}, 'registered_name': None, 'name': 'input_1', 'inbound_nodes': []}, {'module': 'keras.layers', 'class_name': 'ZeroPadding2D', 'config': {'name': 'conv1_pad', 'trainable': False, 'dtype': 'float32', 'padding': [[3, 3], [3, 3]], 'data_format': 'channels_last'}, 'registered_name': None, 'build_config': {'input_shape': [None, 224, 224, 3]}, 'name': 'conv1_pad', 'inbound_nodes': [[['input_1', 0, 0, {}]]]}, {'module': 'keras.layers', 'class_name': 'Conv2D', 'config': {'name': 'conv1_conv', 'trainable': False, 'dtype': 'float32', 'fi

In [ ]:
# Helper functions to load images
def load_single_image(path, size):
    """Load and preprocess a single image using PIL for TIFF compatibility."""
    with Image.open(path) as img:
        img = img.convert('RGB').resize((size, size))
        return np.array(img, dtype=np.float32) / 255.0

def _load_image_py(path):
    """Helper for tf.numpy_function to decode bytes -> path string."""
    if isinstance(path, (bytes, np.bytes_)):
        path = path.decode('utf-8')
    elif isinstance(path, np.ndarray):
        path = path.item()
        if isinstance(path, (bytes, np.bytes_)):
            path = path.decode('utf-8')
    return load_single_image(path, IMAGE_SIZE)

def build_classifier_dataset(data, batch_size, training=False):
    """Build a tf.data.Dataset for classification."""
    paths = [p for p, _ in data]
    labels = [l for _, l in data]
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    
    if training:
        shuffle_buf = min(len(paths), 4096)
        dataset = dataset.shuffle(buffer_size=shuffle_buf, reshuffle_each_iteration=True)

    def _tf_parse(path, label):
        image = tf.numpy_function(func=_load_image_py, inp=[path], Tout=tf.float32)
        image.set_shape((IMAGE_SIZE, IMAGE_SIZE, 3))
        label = tf.cast(label, tf.float32)
        label = tf.reshape(label, [1])
        return image, label

    dataset = dataset.map(_tf_parse, num_parallel_calls=tf.data.AUTOTUNE)
    dataset = dataset.batch(batch_size, drop_remainder=False)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)
    return dataset

print("✅ Helper functions defined")

In [ ]:
# Load test data paths
valid_exts = ['.tif', '.jpg', '.png', '.jpeg', '.bmp']

test_au = [str(f) for f in (BASE_PATH / 'test' / 'Au').glob('*') if f.suffix.lower() in valid_exts]
test_tp = [str(f) for f in (BASE_PATH / 'test' / 'Tp').glob('*') if f.suffix.lower() in valid_exts]

print(f"Test Authentic images: {len(test_au)}")
print(f"Test Tampered images: {len(test_tp)}")
print(f"Total test images: {len(test_au) + len(test_tp)}")

# Create labeled test data: 0 = Authentic, 1 = Tampered
test_data = [(p, 0) for p in test_au] + [(p, 1) for p in test_tp]

# Build test dataset
test_ds = build_classifier_dataset(test_data, BATCH_SIZE, training=False)
print(f"\n✅ Test dataset created")

In [ ]:
# Run predictions on test dataset
print("Running predictions on test data...")
y_true = []
y_pred = []
y_prob = []

for images, labels in test_ds:
    preds = classifier_model.predict(images, verbose=0)
    y_true.extend(labels.numpy().flatten())
    y_prob.extend(preds.flatten())
    y_pred.extend((preds > 0.5).astype(int).flatten())

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_prob = np.array(y_prob)

print(f"✅ Predictions complete!")
print(f"   Total samples: {len(y_true)}")

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Authentic', 'Tampered'],
            yticklabels=['Authentic', 'Tampered'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('ResNet-50 Classifier Confusion Matrix (15 Epochs)')
plt.tight_layout()
plt.show()

# Classification Report
print("\n" + "="*60)
print("CLASSIFICATION REPORT")
print("="*60)
print(classification_report(y_true, y_pred, target_names=['Authentic', 'Tampered']))

In [ ]:
# Additional metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)

print("="*60)
print("MODEL PERFORMANCE SUMMARY")
print("="*60)
print(f"   Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"   Precision: {precision:.4f}")
print(f"   Recall:    {recall:.4f}")
print(f"   F1-Score:  {f1:.4f}")
print(f"   ROC-AUC:   {roc_auc:.4f}")
print("="*60)

In [ ]:
# ROC Curve
from sklearn.metrics import roc_curve

fpr, tpr, thresholds = roc_curve(y_true, y_prob)

plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1, label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - ResNet-50 Classifier (15 Epochs)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()